# Tutorial 4.5: Analyse a cloudy retrieval

Load a cloudy MCMC retrieval and inspect temperature structure, cloud parameters, and model spectra.

## Before you start

A compatible Brewster environment and matching MCMC results, configuration, model arguments, opacity arrays, and cloud-data products.

Review the input paths before running cells. Run cells in order in a fresh kernel. These notebooks retain outputs from earlier runs; a displayed result does not establish that the current setup has been executed or validated.

## Expected outputs

P–T and cloud-location plots, mass and radius estimates, posterior plots, spectra, and walker traces.

## Interpretation

This notebook loads MCMC results. Posterior and likelihood select samples separately; BIC is a sample-based approximation and requires the assumptions explained in Section 2.

Use the repository root for local Brewster imports and relative `data/` and `examples/` paths. If a repository-path setup cell is provided, run it before the imports. Set downloaded opacity/cloud paths and result paths explicitly for your installation.

In [ ]:
from pathlib import Path
import os
import sys

# Use one working directory for Brewster imports and repository data paths.
project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "utils.py").is_file() and (candidate / "retrieval_run.py").is_file():
        project_root = candidate
        break
else:
    raise FileNotFoundError("Start this notebook from within the Brewster repository.")

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Brewster repository:", project_root)

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mgimg
import matplotlib.colors as colors
import scipy as sp
import numpy as np
import emcee
import os 
from collections import namedtuple
import settings
import Priors
import utils
import test_module
import retrieval_run
import corner
import pickle as pickle
import TPmod
from specops import proc_spec
from IPython.display import display
%matplotlib inline

## 1. Load retrieval products

In [ ]:
path = "/Users/kieracharley/Downloads/Retrievals/DBACK1200KEMC504/"

runname = "DBACK1200KEMC504"

# OK finish? 1 for yes, 0 for no.
fin = 1
flatendchain, flatendprobs,ndim = utils.get_endchain(runname,fin,path)
if flatendchain.ndim != 2 or len(flatendchain) == 0:
    raise ValueError("Retained MCMC samples must be a nonempty two-dimensional array.")
if flatendprobs.shape != (len(flatendchain),):
    raise ValueError("Retained log probabilities do not match the samples.")
if not np.isfinite(flatendchain).all() or not np.isfinite(flatendprobs).all():
    raise ValueError("Investigate non-finite retained samples or log-posterior values.")
map_sample_index = np.argmax(flatendprobs)
theta_map_sample = flatendchain[map_sample_index]
max_log_posterior_sample = flatendprobs[map_sample_index]
samples = flatendchain

argfile =path+runname+"_runargs.pic"
runargs = utils.pickle_load(argfile)
opacityfile =path+runname+"_opacities.pic"
opacities = utils.pickle_load(opacityfile)

cloudfile =path+runname+"_cloudata.pic"
cloudata = utils.pickle_load(cloudfile)

settings.init(runargs)
settings.linelist=opacities[0]
settings.cia=opacities[1]
settings.cloudata=cloudata


with open(path+runname+'_configs.pic', 'rb') as file:
    configs= pickle.load(file)
    
re_params=configs['re_params']
model_config_instance=configs['model_config']
if re_params.samplemode.lower() != "mcmc":
    raise ValueError("This notebook loads MCMC chains, not MultiNest output.")

## 2. Distinguish posterior, likelihood, and BIC

For MCMC, `flatendprobs` contains log prior + log likelihood. The legacy `get_endchain` console messages call these values "maximum likelihood", but they are log posterior. `theta_map_sample` is the highest-posterior retained sample, an approximation to the MAP rather than an optimized MAP solution. It is used for the parameter display, mass/radius calculation, and representative spectrum below.

For bounds-only MCMC priors, valid samples have zero log prior, so the next calculation uses `flatendprobs` directly as the data log likelihood. PT type 1 (temperature smoothing and gamma) and parameters with `truncated_gaussian` priors have varying log priors: in those cases, it subtracts the evaluated log prior from each saved log posterior. Neither branch repeats the forward model. This requires the same prior implementation and configuration as the saved retrieval. `theta_ml_sample` is the highest-likelihood retained sample; it can differ from `theta_map_sample`. Neither selection establishes the continuous maximum likelihood.

BIC is defined using the maximum **data likelihood**, not the posterior. We report only a sample-based approximation, `BIC_sample`, which can overestimate BIC if the retained samples miss the likelihood maximum. The data count follows the likelihood's every-third-point selection for uniform FWHM, or all points for an R file. The parameter count excludes the P–T smoothing hyperparameter `gamma`, which enters the MCMC prior rather than the data likelihood; check for any other inactive parameters in custom configurations. BIC's usual regular-model and independent-data assumptions still need assessment before model comparison.

This calculation is restricted to MCMC: the MultiNest `lnlike` branch can include smoothing-prior terms, so its return value cannot automatically be treated as a pure data likelihood.

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # assuming you already have:
# # flatendchain, flatendprobs, ndim, all_params

# n_params = len(all_params)
# fig, axes = plt.subplots(n_params, 1, figsize=(10, 2*n_params), sharex=True)

# for i in range(n_params):
#     ax = axes[i]
#     ax.plot(flatendchain[5000:, i], lw=0.3, alpha=0.7)
#     ax.set_ylabel(all_params[i])
#     if i == 0:
#         ax.set_title("MCMC Chain Traces for Each Parameter")

# axes[-1].set_xlabel("Iteration")
# plt.tight_layout()
# plt.show()

In [ ]:
args_instance=settings.runargs
all_params,all_params_values =utils.get_all_parametres(re_params.dictionary)

params_master = namedtuple('params',all_params)
params_instance = params_master(*theta_map_sample)
params_instance

In [ ]:
if re_params.samplemode.lower() != "mcmc":
    raise ValueError("This likelihood/BIC calculation is for MCMC configurations only.")

mc_init_dis, _, _ = utils.get_dis_range_priors(re_params.dictionary)
has_truncated_gaussian = any(
    prior is not None and prior[0] == "truncated_gaussian"
    for prior in mc_init_dis.values()
)

sample_log_likelihood = flatendprobs.copy()
if runargs.proftype == 1 or has_truncated_gaussian:
    # These priors vary between samples; subtract them from the saved posterior.
    for i, theta in enumerate(samples):
        log_prior = Priors.Priors(theta, re_params, runargs).priors
        sample_log_likelihood[i] -= log_prior

if not np.all(np.isfinite(sample_log_likelihood)):
    raise ValueError("Retained data log likelihoods contain non-finite entries.")

ml_sample_index = np.argmax(sample_log_likelihood)
theta_ml_sample = samples[ml_sample_index]
max_log_likelihood_sample = sample_log_likelihood[ml_sample_index]
log_likelihood_at_map_sample = sample_log_likelihood[map_sample_index]

n_data = runargs.obspec[0, ::3].size if runargs.fwhm is not None else runargs.obspec.shape[1]
k_likelihood = ndim - int(runargs.proftype in (1, 77))
BIC_sample = -2.0 * max_log_likelihood_sample + k_likelihood * np.log(n_data)

print("Highest retained log posterior:", max_log_posterior_sample)
print("Data log likelihood at highest-posterior sample:", log_likelihood_at_map_sample)
print("Highest retained data log likelihood:", max_log_likelihood_sample)
print("Posterior and likelihood select the same sample:", map_sample_index == ml_sample_index)
print("Sample-based BIC approximation:", BIC_sample)
print("Likelihood parameter count and data count:", k_likelihood, n_data)

In [ ]:
re_params.dictionary['pt']['params'].keys()

## 3. Reconstruct the P–T profile

Evaluate P–T profiles for the retained samples and calculate the median and percentile intervals at each pressure.

In [ ]:
all_params,all_params_values =utils.get_all_parametres(re_params.dictionary) 
params_master = namedtuple('params',all_params)
params_instance = params_master(*theta_map_sample)

intemp_keys = list(re_params.dictionary['pt']['params'].keys())

if runargs.proftype==1 or runargs.proftype==77:
    p_index_first=params_instance._fields.index(intemp_keys[1]) 
else:
    p_index_first=params_instance._fields.index(intemp_keys[0]) 
    
p_index_last=params_instance._fields.index(intemp_keys[-1])+1

Tsamples = samples[:,p_index_first:p_index_last]
nsamps = Tsamples.shape[0]
Tprofs = np.empty([64,Tsamples.shape[0]])
for i in range(0,nsamps):
    Tprofs[:,i] = TPmod.set_prof(runargs.proftype,runargs.coarsePress,runargs.press,Tsamples[i,:])
    
Tlays = np.empty([64,5])
for i in range(0,64):
    junk = Tprofs[i,:]
    junk2 = np.percentile(junk, [2.4,16, 50, 84,97.6],axis=0)
    junk3 = np.array(junk2)
    Tlays[i,:] = junk3[:]

In [ ]:
# Temperature profile at the highest-posterior retained sample 

bestT = TPmod.set_prof(runargs.proftype,runargs.coarsePress,runargs.press,theta_map_sample[p_index_first:p_index_last])

In [ ]:
# Update with your exact cloud(s) species, eg. Na2S, KCl, SiO2, etc.

#cloudlist_slab_Mg2SiO4=list(re_params.dictionary["cloud"]["patch 1"]['Mie scattering cloud slab--Mg2SiO4.mieff']['params'])
cloudlist_slab_MgSiO3=list(re_params.dictionary["cloud"]["patch 1"]['Mie scattering cloud slab--MgSiO3.mieff']['params'])
cloudlist_deck=list(re_params.dictionary["cloud"]["patch 1"]['Mie scattering cloud deck--Fe.mieff']['params'])
#cloudlist_slab_Mg2SiO4,cloudlist_slab_MgSiO3,cloudlist_deck
cloudlist_slab_MgSiO3,cloudlist_deck

In [ ]:
# uncomment if deck cloud is in your retrieval

cloudsamples_deck = samples[:,params_instance._fields.index(cloudlist_deck[0]):params_instance._fields.index(cloudlist_deck[-1])+1]
cloud_Top_deck  = np.percentile(cloudsamples_deck [:, 0], [16, 50, 84])
cloud_height_deck  = np.percentile(cloudsamples_deck [:, 1], [16, 50, 84])

In [ ]:
# get the slab cloud
#cloudtemp_slab_Mg2SiO4 = samples[:,params_instance._fields.index(cloudlist_slab_Mg2SiO4[0]):params_instance._fields.index(cloudlist_slab_Mg2SiO4[-1])+1]
#cloudsamples_slab_Mg2SiO4 = np.empty_like(cloudtemp_slab_Mg2SiO4)
#cloudsamples_slab_Mg2SiO4[:,0] = cloudtemp_slab_Mg2SiO4[:,0]
#cloudsamples_slab_Mg2SiO4[:,1] = cloudtemp_slab_Mg2SiO4[:,1]  #+ cloudtemp_slab[:,2]
#cloudsamples_slab_Mg2SiO4[:,2] = cloudtemp_slab_Mg2SiO4[:,2]
#cloudsamples_slab_Mg2SiO4[:,3] = cloudtemp_slab_Mg2SiO4[:,3]
#cloudsamples_slab_Mg2SiO4[:,4] = cloudtemp_slab_Mg2SiO4[:,4]


# get the slab cloud
cloudtemp_slab_MgSiO3 = samples[:,params_instance._fields.index(cloudlist_slab_MgSiO3[0]):params_instance._fields.index(cloudlist_slab_MgSiO3[-1])+1]
cloudsamples_slab_MgSiO3 = np.empty_like(cloudtemp_slab_MgSiO3)
cloudsamples_slab_MgSiO3[:,0] = cloudtemp_slab_MgSiO3[:,0]
cloudsamples_slab_MgSiO3[:,1] = cloudtemp_slab_MgSiO3[:,1]  #+ cloudtemp_slab[:,2]
cloudsamples_slab_MgSiO3[:,2] = cloudtemp_slab_MgSiO3[:,2]
cloudsamples_slab_MgSiO3[:,3] = cloudtemp_slab_MgSiO3[:,3]
cloudsamples_slab_MgSiO3[:,4] = cloudtemp_slab_MgSiO3[:,4]



#cloud_Base_slab_Mg2SiO4 = np.percentile(cloudsamples_slab_Mg2SiO4[:, 1], [16, 50, 84])
#cloud_height_slab_Mg2SiO4 = np.percentile(cloudsamples_slab_Mg2SiO4[:, 2], [16, 50, 84])
#cloud_Tau_slab_Mg2SiO4 = np.percentile(cloudsamples_slab_Mg2SiO4[:, 0], [16, 50, 84])



cloud_Base_slab_MgSiO3 = np.percentile(cloudsamples_slab_MgSiO3[:, 1], [16, 50, 84])
cloud_height_slab_MgSiO3 = np.percentile(cloudsamples_slab_MgSiO3[:, 2], [16, 50, 84])
cloud_Tau_slab_MgSiO3 = np.percentile(cloudsamples_slab_MgSiO3[:, 0], [16, 50, 84])

In [ ]:
# Plot the posterior P–T intervals, representative profile, and cloud locations
plt.rc('font',family='Times New Roman')
fig=plt.figure(dpi=320)

prf = fig.add_axes([0.1, 0.1, 0.65, 0.85])

prf.axis([0., 3500.,3.0,-4.0])
plt.ylabel(r'log(P / bar)')
plt.xlabel('T / K')


# Plotting deck cloud (Fe)

logP = np.log10(runargs.press)
cld1 = fig.add_axes([0.75,0.1,0.07,0.85],xticklabels=[],xlim=[0,1])
cld1.axis([0,1,3,-4])
cld1.tick_params(axis='y',direction='in',left=True,right=True,labelleft=False,labelright=False)

BesTprof = TPmod.set_prof(runargs.proftype,runargs.coarsePress,runargs.press,params_instance[p_index_first:p_index_last])


npts = 10

cmap = mpl.cm.Blues
y = np.linspace(cloud_Top_deck[1] - cloud_height_deck[1],cloud_Top_deck[1],npts)
normalize = mpl.colors.Normalize(vmin=y.min(), vmax=y.max()+2)
for j in range(0,npts-1):
         cld1.fill_between([0.,1.],y[j],y[j+1],facecolor=cmap(normalize(y[j])), linewidth=0.0)

cld1.fill_between([0.,1.],3.0,cloud_Top_deck[1],facecolor='blue',alpha=0.3)
cld1.fill_between([0,1.],cloud_Top_deck[2],cloud_Top_deck[0],facecolor='grey',alpha=0.8)
cld1.plot([0,1],[cloud_Top_deck[1] -cloud_height_deck[1],cloud_Top_deck[1] - cloud_height_deck[1]],'k--',linewidth=1.0)
cld1.set_title("deck")
cld1.text(0.3,2.5,r"$\rm Fe$",fontsize=7)


# Plotting slab cloud (Mg2SiO4)

#cld2 = fig.add_axes([0.82,0.1,0.07,0.85],xticklabels=[],xlim=[0,1])
#cld2.axis([0,1,3,-4])
#cld2.tick_params(axis='y',direction='in',left=True,right=True,labelleft=False,labelright=False)

#cld2.fill_between([0,1.],cloud_Base_slab_Mg2SiO4[1],cloud_Base_slab_Mg2SiO4[1] - cloud_height_slab_Mg2SiO4[1],facecolor='blue',alpha=0.5)
#cld2.fill_between([0,1.],cloud_Base_slab_Mg2SiO4[2],cloud_Base_slab_Mg2SiO4[1] - cloud_height_slab_Mg2SiO4[2],facecolor='grey',alpha=0.5)
#cld2.set_title("slab")
#cld2.text(0.10,2.5,r"$\rm Mg_{2}SiO_{4}$",fontsize=6)


# Plotting slab cloud (MgSiO3)

cld3 = fig.add_axes([0.82,0.1,0.07,0.85],xticklabels=[],xlim=[0,1])
cld3.axis([0,1,3,-4])
cld3.tick_params(axis='y',direction='in',left=True,right=True,labelleft=False,labelright=True)

cld3.fill_between([0,1.],cloud_Base_slab_MgSiO3[1],cloud_Base_slab_MgSiO3[1] - cloud_height_slab_MgSiO3[1],facecolor='blue',alpha=0.5)
cld3.fill_between([0,1.],cloud_Base_slab_MgSiO3[2],cloud_Base_slab_MgSiO3[1] - cloud_height_slab_MgSiO3[2],facecolor='grey',alpha=0.5)
cld3.set_title("slab")
cld3.text(0.10,2.5,r"$\rm MgSiO_{3}$",fontsize=6)




d1, = prf.plot(Tlays[:,2],logP,'k-',label='median')
prf.fill_betweenx(logP,Tlays[:,1], Tlays[:,3], facecolor='red', alpha=0.3)
prf.fill_betweenx(logP,Tlays[:,0], Tlays[:,4], facecolor='red', alpha=0.1)

d2, = prf.plot(BesTprof,logP,'k--',label='highest-posterior sample')



pero = 10.0**4 / (5.125 - 0.277*logP - 0.554*0.0)

qtz = 10.0**4 / (6.12 - 0.34*logP - 0.69*0.0)
qtz_alt = 10.0**4 / (6.27 - 0.34*logP - 0.69*0.0)


enst = 10.0**4/(6.26 - 0.35*logP-0.70*0.0)
fost = 10.0**4/(5.89 - 0.37*logP-0.73*0.0)
iron = 10.0**4/(5.44 - 0.48*logP-0.48*0.0)
cr =  10.0**4/(6.528 - 0.491*logP-0.491*0.0)
al2o3 = 10.0**4 / (5.0139 - 0.21794*(logP) + 2.2636E-03*(logP)**2.0 - 0.580*0.0)

c1, = prf.plot(enst,logP,'--',color='blue',linewidth=1.5, label='MgSiO$_3$')
c2, = prf.plot(qtz,logP,'--',color='magenta',linewidth=1.5, label='SiO$_2$')
c3, = prf.plot(fost,logP,'--',color='pink',linewidth=1.5,label='Mg$_2$SiO$_4$')
c4, = prf.plot(iron,logP,'--',color='orange',linewidth=1.5, label='Fe')
c5, = prf.plot(pero,logP,'--',color='purple',linewidth=1.5, label='CaTiO$_3$')
c6, = prf.plot(al2o3,logP,'--',color='red',linewidth=1.5, label='Al$_2$O$_3$')

prf.legend(handles=[d1,d2,c1,c2,c3,c4,c5,c6],prop={'size':8}) 


plt.tight_layout()
plt.show()

## 4. Derive mass and radius

In [ ]:
D = 3.086e+16 * model_config_instance.dist

r2d2_index=params_instance._fields.index('r2d2')
r2d2 =theta_map_sample[r2d2_index]


logg_index=params_instance._fields.index('logg')
logg =theta_map_sample[logg_index]


if (r2d2 > 0.):
    R = np.sqrt(r2d2) * D
g = (10.**logg)/100.
M = (R**2 * g/(6.67E-11))/1.898E27
Rj = R / 69911.e3
print(M,Rj)

In [ ]:
r2d2 = samples[:,r2d2_index,]
logg = samples[:,logg_index]


R = np.sqrt(r2d2) * D
g = (10.**logg)/100.
M = (R**2 * g/(6.67E-11))/1.898E27
Rj = R / 69911.e3
Msig = np.percentile(M,[16,50,84])
Rsig =  np.percentile(Rj,[16,50,84])
print('M = ',Msig[1], '+ ', Msig[2] - Msig[1], ' - ', Msig[1] - Msig[0])
print('R = ',Rsig[1], '+ ', Rsig[2] - Rsig[1], ' - ', Rsig[1] - Rsig[0])

## 5. Plot parameter distributions

Use a corner plot to inspect marginal distributions and parameter correlations. Assess sampling quality separately before reporting intervals.

In [ ]:
fig = corner.corner(samples,scale_hist=False,plot_datapoints =False,\
                    labels=all_params,\
                    quantiles=[0.16, 0.5, 0.84],show_titles=True, title_kwargs={"fontsize": 20},\
                    label_kwargs={"fontsize": 20},color='deeppink')
#plt.savefig('DBACK1200KEMC504_cornerplot.png', dpi ='figure', format = 'png')

## 6. Compare model and observed spectra

Generate a representative spectrum at the highest-posterior retained sample and inspect the distribution of model spectra. This curve is not labelled as a maximum-likelihood fit.

The existing spectral-draw loop passes the representative parameter vector to instrumental processing instead of the individual draw. If instrumental nuisance parameters vary, its intervals do not fully propagate their uncertainty. Review that separate limitation before reporting these intervals.

In [ ]:
# get diagnostics along with the spectrum
gnostics = 0
# Now run the model again to get your model spectrum and process to make it look like the data

# leave commented for a cloudy retrieval
#settings.cloudata = np.asfortranarray(np.zeros([4,102]))
trimspec, photspec, tauspec,cfunc = test_module.modelspec(theta_map_sample,re_params,runargs,gnostics)
wave,topspec=proc_spec(inputspec=trimspec, theta=params_instance, re_params=re_params, args_instance=args_instance, do_scales=args_instance.do_scales, do_shift=args_instance.do_shift)

In [ ]:
# Now grab 100 random draws from the posterior
pltspec = np.zeros((100,runargs.obspec[0,:].size))
samp= np.empty(ndim)
samples = flatendchain
sid = np.zeros(100)
for i in range (0,100):
    sid[i]= np.random.randint(0,high = len(samples))
    samp = samples[int(sid[i]),:]
    
    trimspec, photspec, tauspec,cfunc = test_module.modelspec(samp,re_params,runargs,gnostics)
    pltspec[i,:] = proc_spec(inputspec=trimspec, theta=params_instance, re_params=re_params, args_instance=args_instance, do_scales=args_instance.do_scales, do_shift=args_instance.do_shift)[1]

# get the intervals for the distribution of model spectra
specdist = np.empty([runargs.obspec[0].size,5])
for i in range(0,runargs.obspec[0].size):
    junk = pltspec[:,i]
    junk2 = np.percentile(junk, [2.4,16, 50, 84,97.6],axis=0)
    junk3 = np.array(junk2)
    specdist[i,:] = junk3[:]

In [ ]:
# plot the spectra
plt.rc('font',family='Times New Roman')
fig=plt.figure(dpi=120)


d1, = plt.plot(runargs.obspec[0,:],runargs.obspec[1,:],'k-',label = "data")
plt.fill_between(runargs.obspec[0], runargs.obspec[1,:] - runargs.obspec[2,:],runargs.obspec[1,:] + runargs.obspec[2,:],facecolor='grey',alpha=0.5)

#t1, = plt.plot(runargs.obspec[0,:],topspec,'g-',linewidth=1, label = runname+" highest-posterior sample")
t1, = plt.plot(runargs.obspec[0,:],topspec,c='deeppink',linewidth=1, label = runname+" highest-posterior sample")


#r1, = plt.plot(runargs.obspec[0],specdist[:,2],'y-',linewidth=0.5, label = "median")
r1, = plt.plot(runargs.obspec[0],specdist[:,2],c='lightpink',linewidth=0.5, label = "median")
#plt.fill_between(obspec[0],specdist[:,0],specdist[:,4],facecolor='red',alpha=0.2)
plt.fill_between(runargs.obspec[0],specdist[:,1],specdist[:,3],facecolor='indigo',alpha=0.5)
#plt.fill_between(obspec[0,:],obspec[1,:]-obspec[2,:],obspec[1,:]+obspec[2,:],facecolor='red',alpha=0.2)


plt.legend(handles=[d1,t1,r1])


plt.ylabel(r'$ F_{\lambda}$ / $Wm^{-2} \mu m^{-1}$')
plt.xlabel('Wavelength / $\mu m$')
#plt.savefig(runname+"_SPAG_SPEC.png",format='png', dpi=320)

In [ ]:
# Walker trace plots

with open(path + runname + '_snapshot.pic', 'rb') as f:
    pos, lnprob = pickle.load(f)

nwalkers, nsteps, ndim = pos.shape
print(f"Chain shape: {pos.shape} (walkers, steps, parameters)")

param_index = 2 
plt.figure(figsize=(10, 6))
for w in range(nwalkers):
    plt.plot(pos[w, :, param_index], alpha=0.3, lw=0.5)
plt.xlabel("Step")
plt.ylabel(all_params[param_index])
plt.title(f"Trace plot for {all_params[param_index]}")
plt.tight_layout()
plt.show()

## Checks and next steps

This notebook loads MCMC results. Posterior and likelihood select samples separately; BIC is a sample-based approximation and requires the assumptions explained in Section 2.

Keep the input configuration and data provenance with any exported figures or tables. Continue with [Priors and likelihood](../priors_likelihood.rst) for the corresponding framework guide.